# Positive Definite Matrices — Exercises

8 graded exercises from PD verification to Cholesky-based VAE sampling.

| Format | Description |
|--------|-------------|
| **Problem** | Markdown cell with task description |
| **Your Solution** | Code cell with scaffolding |
| **Solution** | Code cell with reference solution and checks |

### Difficulty Levels

| Level | Exercises | Focus |
|-------|-----------|-------|
| ★ | 1–3 | Core mechanics |
| ★★ | 4–6 | Deeper theory |
| ★★★ | 7–8 | AI / ML applications |

### Topic Map

| Topic | Exercise |
|-------|----------|
| PD definition & Loewner order | 1 |
| Cholesky algorithm | 2 |
| Sylvester's criterion | 3 |
| LDLᵀ factorization | 4 |
| Schur complement & block PD | 5 |
| Log-determinant & gradient | 6 |
| Gram matrices & kernel regression | 7 |
| VAE reparameterization trick | 8 |

In [ ]:
import numpy as np
import numpy.linalg as la
import scipy.linalg as sla

try:
    import matplotlib.pyplot as plt
    HAS_MPL = True
except ImportError:
    HAS_MPL = False

np.set_printoptions(precision=6, suppress=True)
np.random.seed(42)

def header(title):
    print('\n' + '=' * len(title))
    print(title)
    print('=' * len(title))

def check_close(name, got, expected, tol=1e-8):
    ok = np.allclose(got, expected, atol=tol, rtol=tol)
    print(f"{'PASS' if ok else 'FAIL'} — {name}")
    if not ok:
        print(f'  expected: {expected}')
        print(f'  got:      {got}')
    return ok

def check_true(name, cond):
    print(f"{'PASS' if cond else 'FAIL'} — {name}")
    return cond

def is_pd(A, tol=1e-10):
    return np.all(la.eigvalsh(A) > tol)

def is_psd(A, tol=1e-10):
    return np.all(la.eigvalsh(A) >= -tol)

def log_det_chol(A):
    L = la.cholesky(A)
    return 2.0 * np.sum(np.log(np.diag(L)))

print('Setup complete.')

---

## Exercise 1 ★ — PD Axioms for $A^\top A + \lambda I$

Let $A \in \mathbb{R}^{m \times n}$ with $m \geq n$ and $\lambda > 0$. Define $B = A^\top A + \lambda I$.

**(a)** Prove $B \succ 0$ by computing $\mathbf{x}^\top B \mathbf{x}$ for $\mathbf{x} \neq \mathbf{0}$.

**(b)** What is the smallest eigenvalue of $B$? (Hint: use eigenvalues of $A^\top A$.)

**(c)** Show $B^{-1} \preceq \frac{1}{\lambda}I$ using the Loewner order.

**(d)** Verify numerically for a random $5 \times 3$ matrix $A$ and $\lambda = 0.5$.

**(e)** **AI connection:** Identify $B = A^\top A + \lambda I$ as the ridge regression normal equations matrix.

In [ ]:
# Exercise 1: Your Solution

np.random.seed(42)
m, n = 5, 3
lam = 0.5
A = np.random.randn(m, n)
B = A.T @ A + lam * np.eye(n)

# (b) Smallest eigenvalue of B
min_eig_B = None  # YOUR CODE HERE

# (c) Verify B^{-1} preceq (1/lam) * I
B_inv = None  # YOUR CODE HERE
bound = None  # YOUR CODE HERE (the bound matrix)
loewner_ok = None  # YOUR CODE HERE: check that bound - B_inv is PSD

print(f'B min eigenvalue: {min_eig_B}')
print(f'Loewner holds: {loewner_ok}')

In [ ]:
# Exercise 1: Solution

np.random.seed(42)
m, n = 5, 3
lam = 0.5
A = np.random.randn(m, n)
B = A.T @ A + lam * np.eye(n)

header('Exercise 1: PD Axioms for A^T A + lambda I')

# (a) Quadratic form
# x^T B x = x^T A^T A x + lambda ||x||^2 = ||Ax||^2 + lambda||x||^2 >= lambda||x||^2 > 0
# (since lambda > 0 and x != 0 => ||x||^2 > 0)
check_true('(a) B is PD', is_pd(B))

# (b) Smallest eigenvalue
sigma_min_sq = la.eigvalsh(A.T @ A).min()  # = sigma_min(A)^2
min_eig_B_theory = sigma_min_sq + lam
min_eig_B_actual = la.eigvalsh(B).min()
check_close('(b) min eigenvalue = sigma_min(A)^2 + lambda',
            min_eig_B_actual, min_eig_B_theory)

# (c) Loewner order: B^{-1} preceq (1/lam) * I
# Equivalent: (1/lam)*I - B^{-1} succeq 0
B_inv = la.inv(B)
diff = (1/lam) * np.eye(n) - B_inv
check_true('(c) (1/lam)*I - B^{-1} is PSD', is_psd(diff))

# (d) Numerical verification
x = np.random.randn(n)
qform = x @ B @ x
lower_bound = lam * np.sum(x**2)
check_true('(d) x^T B x >= lambda ||x||^2', qform >= lower_bound - 1e-12)
print(f'  x^T B x = {qform:.4f}, lambda||x||^2 = {lower_bound:.4f}')

# (e) Ridge regression connection
y = np.random.randn(m)
beta_ridge = la.solve(B, A.T @ y)
beta_ols   = la.lstsq(A.T @ A, A.T @ y, rcond=None)[0]
print(f'\nRidge solution:  {beta_ridge.round(4)}')
print(f'OLS (lambda=0):  {beta_ols.round(4)}')
print("Takeaway: B = A^T A + lam*I is always PD (lambda > 0), making ridge regression" \
      " numerically stable regardless of rank(A).")

---

## Exercise 2 ★ — Cholesky from Scratch

Implement the Cholesky algorithm using only basic NumPy operations.

**(a)** Implement `cholesky_impl(A)` that returns lower triangular $L$ with $A = LL^\top$ or raises `ValueError` if $A$ is not PD.

**(b)** Test on $A = \begin{pmatrix}4 & 12 & -16 \\ 12 & 37 & -43 \\ -16 & -43 & 98\end{pmatrix}$. Verify $LL^\top = A$.

**(c)** Test on the indefinite matrix $A = \begin{pmatrix}1 & 3 \\ 3 & 1\end{pmatrix}$ — should raise `ValueError`.

**(d)** Count the floating-point operations (flops) for your $n=3$ example.

**(e)** Compare speed with `np.linalg.cholesky` for $n = 200$.

In [ ]:
# Exercise 2: Your Solution

def cholesky_impl(A):
    # YOUR CODE HERE
    pass

A_test = np.array([[4., 12., -16.],
                   [12., 37., -43.],
                   [-16., -43., 98.]])

L = cholesky_impl(A_test)
print('L:', L)
print('L @ L.T:', L @ L.T if L is not None else None)

In [ ]:
# Exercise 2: Solution

def cholesky_impl(A):
    """Cholesky factorization from scratch."""
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.zeros((n, n))
    for j in range(n):
        s = A[j, j] - np.sum(L[j, :j]**2)
        if s <= 0:
            raise ValueError(f'Not PD: non-positive pivot {s:.4e} at column {j}')
        L[j, j] = np.sqrt(s)
        for i in range(j+1, n):
            L[i, j] = (A[i, j] - np.sum(L[i, :j] * L[j, :j])) / L[j, j]
    return L

header('Exercise 2: Cholesky from Scratch')

# (b)
A_test = np.array([[4., 12., -16.],
                   [12., 37., -43.],
                   [-16., -43., 98.]])
L = cholesky_impl(A_test)
print('L:')
print(L)
check_close('(b) L @ L.T = A', L @ L.T, A_test)
check_close('(b) Matches np.linalg.cholesky', L, la.cholesky(A_test))

# (c)
A_indef = np.array([[1., 3.], [3., 1.]])
try:
    cholesky_impl(A_indef)
    check_true('(c) Should have failed', False)
except ValueError as e:
    check_true('(c) Correctly raises ValueError for indefinite matrix', True)
    print(f'  Error: {e}')

# (d) Flop count for n=3
# j=0: 1 sqrt, 0 mults -> 1 flop for diag; 2 divides for off-diag -> ~3 flops
# j=1: 1 sub + 1 sqrt; 1 mul+sub+div for row 2 -> ~5 flops
# j=2: 2 sub + 1 sqrt -> ~3 flops
# Total: ~11 flops for n=3. Formula: n^3/3 = 9 for n=3 (counting multiplications only)
print(f'\n(d) Theoretical flops n^3/3 for n=3: {3**3/3:.0f}')

# (e) Speed comparison
import time
n = 200
X = np.random.randn(n, n)
A_big = X.T @ X + n * np.eye(n)

t0 = time.perf_counter()
for _ in range(5): L_np = la.cholesky(A_big)
t_np = (time.perf_counter()-t0)/5

t0 = time.perf_counter()
L_ours = cholesky_impl(A_big)
t_ours = time.perf_counter()-t0

print(f'\n(e) n={n}: np.linalg.cholesky={t_np*1000:.2f}ms, our impl={t_ours*1000:.2f}ms')
print(f'Speedup (NumPy/ours): {t_ours/t_np:.1f}x (LAPACK is faster — uses BLAS)')
print("\nTakeaway: Cholesky requires n^3/3 flops — half of LU's 2n^3/3 for symmetric PD systems.")

---

## Exercise 3 ★ — Sylvester's Criterion

Let $A(\alpha) = \begin{pmatrix}1 & \alpha & 0 \\ \alpha & 2 & \alpha \\ 0 & \alpha & 4\end{pmatrix}$.

**(a)** Compute $\Delta_1, \Delta_2, \Delta_3$ as functions of $\alpha$.

**(b)** Find all $\alpha$ for which $A(\alpha) \succ 0$.

**(c)** At the boundary $\alpha$ value, find the null vector of $A(\alpha)$.

**(d)** Plot the minimum eigenvalue of $A(\alpha)$ vs $\alpha$ and mark the PD region.

In [ ]:
# Exercise 3: Your Solution

def A_alpha(alpha):
    return np.array([[1., alpha, 0.],
                     [alpha, 2., alpha],
                     [0., alpha, 4.]])

# (a) Compute leading principal minors as function of alpha
# Delta1 = YOUR CODE HERE
# Delta2 = YOUR CODE HERE
# Delta3 = YOUR CODE HERE

alpha_test = 0.5
A = A_alpha(alpha_test)
delta1 = None  # YOUR CODE HERE
delta2 = None  # YOUR CODE HERE
delta3 = None  # YOUR CODE HERE
print(f'At alpha={alpha_test}: Delta1={delta1}, Delta2={delta2}, Delta3={delta3}')

In [ ]:
# Exercise 3: Solution

def A_alpha(alpha):
    return np.array([[1., alpha, 0.],
                     [alpha, 2., alpha],
                     [0., alpha, 4.]])

header('Exercise 3: Sylvester Criterion for A(alpha)')

# (a) Analytical minors
# Delta1 = 1 (always positive)
# Delta2 = 1*2 - alpha^2 = 2 - alpha^2
# Delta3 = det(A) = 1*(2*4 - alpha^2) - alpha*(alpha*4 - 0) + 0
#        = 8 - alpha^2 - 4*alpha^2 = 8 - 5*alpha^2

alpha_vals = np.linspace(-2.5, 2.5, 1000)
delta2_vals = 2 - alpha_vals**2
delta3_vals = 8 - 5*alpha_vals**2

# PD: both > 0
# Delta2 > 0: |alpha| < sqrt(2)
# Delta3 > 0: |alpha| < sqrt(8/5) = sqrt(1.6)
# Binding: Delta3 (smaller range)
alpha_bd = np.sqrt(8/5)
print(f'(b) PD for |alpha| < sqrt(8/5) = {alpha_bd:.4f}')

# Verify
for alpha in [0, 0.5, alpha_bd - 0.001, alpha_bd + 0.001, 1.5]:
    A = A_alpha(alpha)
    d1 = A[0,0]
    d2 = la.det(A[:2,:2])
    d3 = la.det(A)
    pd = d1 > 0 and d2 > 0 and d3 > 0
    print(f'  alpha={alpha:.4f}: Delta3={d3:.4f}, PD={pd}')

# (c) Null vector at boundary
A_bd = A_alpha(alpha_bd)
eigs, vecs = la.eigh(A_bd)
null_vec = vecs[:, 0]  # smallest eigenvalue ~ 0
print(f'\n(c) At alpha={alpha_bd:.4f}: min eigenvalue={eigs[0]:.2e}')
print(f'  Null vector: {null_vec.round(4)}')
print(f'  A * null_vec: {(A_bd @ null_vec).round(6)}')

# (d) Plot
if HAS_MPL:
    fig, ax = plt.subplots(figsize=(9, 4))
    min_eigs = [la.eigvalsh(A_alpha(a)).min() for a in alpha_vals]
    ax.plot(alpha_vals, min_eigs, color='#0077BB', linewidth=2)
    ax.axhline(0, color='#555555', linestyle='--', linewidth=0.8)
    ax.axvline( alpha_bd, color='#CC3311', linestyle=':', linewidth=1.5, label=f'bound={alpha_bd:.3f}')
    ax.axvline(-alpha_bd, color='#CC3311', linestyle=':', linewidth=1.5)
    ax.fill_between(alpha_vals, min_eigs, 0,
                    where=np.array(min_eigs) > 0, alpha=0.2, color='#009988', label='PD region')
    ax.set_title(r'Min eigenvalue of $A(\alpha)$: PD iff $|\alpha| < \sqrt{8/5}$')
    ax.set_xlabel(r'$\alpha$'); ax.set_ylabel(r'$\lambda_{\min}$')
    ax.legend(); fig.tight_layout(); plt.show()

print("\nTakeaway: Sylvester's criterion reveals the exact parameter range for PD without eigendecomposition.")

---

## Exercise 4 ★★ — LDL$^\top$ Factorization

**(a)** Implement `ldlt_impl(A)` returning $(L, \mathbf{d})$ where $A = LDL^\top$, $L$ unit lower triangular, $D = \text{diag}(\mathbf{d})$.

**(b)** Test on $A = \begin{pmatrix}4 & 2 & 1 \\ 2 & 5 & 3 \\ 1 & 3 & 6\end{pmatrix}$. Verify $LDL^\top = A$.

**(c)** Solve $A\mathbf{x} = (1,2,3)^\top$ using three triangular solves.

**(d)** Show that the LDLᵀ pivots (diagonal of $D$) are all positive for PD matrices.

**(e)** Explain when to prefer LDLᵀ over Cholesky.

In [ ]:
# Exercise 4: Your Solution

def ldlt_impl(A):
    # YOUR CODE HERE
    # Return (L, d) where d = diagonal of D
    pass

A = np.array([[4., 2., 1.], [2., 5., 3.], [1., 3., 6.]])
L, d = ldlt_impl(A) if ldlt_impl(A) is not None else (None, None)
print('L:', L)
print('d:', d)

In [ ]:
# Exercise 4: Solution

def ldlt_impl(A):
    A = np.array(A, dtype=float)
    n = A.shape[0]
    L = np.eye(n)
    d = np.zeros(n)
    for j in range(n):
        v = d[:j] * L[j, :j]
        d[j] = A[j, j] - np.dot(L[j, :j], v)
        if abs(d[j]) < 1e-14:
            raise ValueError(f'Zero pivot at j={j}')
        for i in range(j+1, n):
            L[i, j] = (A[i, j] - np.dot(L[i, :j], v)) / d[j]
    return L, d

header('Exercise 4: LDL^T Factorization')

A = np.array([[4., 2., 1.], [2., 5., 3.], [1., 3., 6.]])

# (a) + (b)
L, d = ldlt_impl(A)
D = np.diag(d)
print('L (unit lower triangular):')
print(L)
print('d (diagonal of D):', d)
check_close('(b) L @ D @ L.T = A', L @ D @ L.T, A)

# (c) Solve Ax = b
b = np.array([1., 2., 3.])
# Step 1: L y = b  (forward substitution)
y = sla.solve_triangular(L, b, lower=True, unit_diagonal=True)
# Step 2: D w = y  (trivial: w_i = y_i / d_i)
w = y / d
# Step 3: L^T x = w  (backward substitution)
x = sla.solve_triangular(L.T, w, lower=False, unit_diagonal=True)

x_direct = la.solve(A, b)
check_close('(c) LDLt solve matches direct', x, x_direct)
print(f'x = {x.round(6)}')

# (d) Pivots for PD
check_true('(d) All pivots > 0 for PD matrix', np.all(d > 0))
print(f'Pivots: {d}')

# (e) Indefinite test
A_indef = np.array([[1., 3.], [3., 1.]])  # indefinite
L_i, d_i = ldlt_impl(A_indef)
print(f'\n(e) Indefinite matrix pivots: {d_i} (some negative -> indefinite)')
print("Prefer LDL^T over Cholesky when: (1) avoiding sqrt, (2) indefinite matrix, (3) integer arithmetic.")
print("\nTakeaway: LDL^T pivots are the 'signature' of the quadratic form; all positive <=> PD.")

---

## Exercise 5 ★★ — Schur Complement and Block PD

Let $M = \begin{pmatrix}A & B \\ B^\top & D\end{pmatrix}$ where
$A = \begin{pmatrix}4 & 1 \\ 1 & 3\end{pmatrix}$,
$B = \begin{pmatrix}1 \\ 2\end{pmatrix}$,
$D = \begin{pmatrix}5\end{pmatrix}$.

**(a)** Compute the Schur complement $S = D - B^\top A^{-1}B$.

**(b)** Use Theorem 6.2 to determine whether $M \succ 0$.

**(c)** Compute $\det M$ via $\det A \cdot \det S$ and verify numerically.

**(d)** Compute $M^{-1}$ using the block inverse formula. Verify $MM^{-1} = I$.

**(e)** Interpret $M$ as a $3 \times 3$ covariance matrix. What is $\text{Var}(X_1 | X_2 = a, X_3 = b)$?

In [ ]:
# Exercise 5: Your Solution

A = np.array([[4., 1.], [1., 3.]])
B = np.array([[1.], [2.]])
D = np.array([[5.]])
M = np.block([[A, B], [B.T, D]])

# (a) Schur complement
S = None  # YOUR CODE HERE

# (b) Is M PD?
M_pd = None  # YOUR CODE HERE

print(f'Schur complement S = {S}')
print(f'M is PD: {M_pd}')

In [ ]:
# Exercise 5: Solution

A = np.array([[4., 1.], [1., 3.]])
B = np.array([[1.], [2.]])
D = np.array([[5.]])
M = np.block([[A, B], [B.T, D]])

header('Exercise 5: Schur Complement and Block PD')

# (a)
A_inv = la.inv(A)
S = D - B.T @ A_inv @ B
print(f'(a) Schur complement S = D - B^T A^{{-1}} B = {S[0,0]:.6f}')

# (b)
A_pd = is_pd(A)
S_pd = S[0,0] > 0
M_pd_schur = A_pd and S_pd
M_pd_direct = is_pd(M)
check_true('(b) Schur criterion matches direct check', M_pd_schur == M_pd_direct)
print(f'  A PD: {A_pd}, S>0: {S_pd}, M PD: {M_pd_schur}')

# (c) Determinant
det_schur  = la.det(A) * S[0,0]
det_direct = la.det(M)
check_close('(c) det via Schur matches det direct', det_schur, det_direct)
print(f'  det(A)={la.det(A):.4f}, det(S)={S[0,0]:.4f}, product={det_schur:.4f}')

# (d) Block inverse
# M^{-1} = [[A^{-1} + A^{-1}B S^{-1} B^T A^{-1}, -A^{-1}B S^{-1}],
#           [-S^{-1} B^T A^{-1},                  S^{-1}        ]]
S_inv = 1.0 / S[0,0]
M11 = A_inv + A_inv @ B @ np.array([[S_inv]]) @ B.T @ A_inv
M12 = -A_inv @ B * S_inv
M21 = -S_inv * B.T @ A_inv
M22 = np.array([[S_inv]])
M_inv = np.block([[M11, M12], [M21, M22]])
check_close('(d) M M^{-1} = I', M @ M_inv, np.eye(3))

# (e) Conditional variance
# Interpreting M as Sigma_{33} with Sigma_{11}=A (2x2), Sigma_{12}=B, Sigma_{22}=D
# Var(X1,X2 | X3) = Schur complement of D in M = A - B * D^{-1} * B^T
Sigma_cond = A - B @ (1.0/D) @ B.T
print(f'\n(e) Cond covariance Sigma_{{12|3}} = A - B D^{{-1}} B^T:')
print(Sigma_cond)
print(f'Conditional variance < marginal: {np.all(np.diag(Sigma_cond) < np.diag(A))}')
print("\nTakeaway: Schur complement = conditional covariance. Conditioning always reduces uncertainty (Loewner order).")

---

## Exercise 6 ★★ — Log-Determinant and Gradient Verification

Let $A = \begin{pmatrix}3 & 1 & 0 \\ 1 & 4 & 2 \\ 0 & 2 & 5\end{pmatrix}$.

**(a)** Compute $\log\det A$ via (i) eigenvalues, (ii) Cholesky diagonal formula, (iii) `np.linalg.det`. All three must agree.

**(b)** Implement `log_det(A)` using Cholesky and compare speed vs eigenvalue method for $n = 100, 500$.

**(c)** Numerically verify $\nabla_A \log\det A = A^{-1}$ using finite differences.

**(d)** Compute the Gaussian log-likelihood using the log-det formula.

In [ ]:
# Exercise 6: Your Solution

A = np.array([[3., 1., 0.], [1., 4., 2.], [0., 2., 5.]])

# (a) Three methods
ld_eig    = None  # YOUR CODE HERE
ld_chol   = None  # YOUR CODE HERE
ld_direct = None  # YOUR CODE HERE

print(f'Eigenvalue method: {ld_eig}')
print(f'Cholesky method:   {ld_chol}')
print(f'Direct det():      {ld_direct}')

In [ ]:
# Exercise 6: Solution

A = np.array([[3., 1., 0.], [1., 4., 2.], [0., 2., 5.]])

header('Exercise 6: Log-Determinant and Gradient')

# (a)
ld_eig    = np.sum(np.log(la.eigvalsh(A)))
ld_chol   = log_det_chol(A)
ld_direct = np.log(la.det(A))

print(f'(a) Eigenvalue: {ld_eig:.8f}')
print(f'    Cholesky:   {ld_chol:.8f}')
print(f'    Direct:     {ld_direct:.8f}')
check_close('(a) All three agree', ld_eig, ld_direct)

# (b) Speed comparison
import time
for n in [100, 500]:
    np.random.seed(42)
    X = np.random.randn(n, n)
    B = X.T @ X + n * np.eye(n)
    
    reps = 10
    t0 = time.perf_counter()
    for _ in range(reps): _ = log_det_chol(B)
    t_chol = (time.perf_counter()-t0)/reps
    
    t0 = time.perf_counter()
    for _ in range(reps): _ = np.sum(np.log(la.eigvalsh(B)))
    t_eig = (time.perf_counter()-t0)/reps
    
    print(f'(b) n={n}: Cholesky={t_chol*1000:.2f}ms  Eigenvalue={t_eig*1000:.2f}ms')

# (c) Gradient verification
h = 1e-5
A_inv = la.inv(A)
fd_grad = np.zeros_like(A)
for i in range(3):
    for j in range(3):
        E = np.zeros((3,3)); E[i,j] = h
        try:
            fd_grad[i,j] = (log_det_chol(A+E) - log_det_chol(A-E)) / (2*h)
        except: pass

err = np.max(np.abs(A_inv - fd_grad))
check_true('(c) FD gradient matches A^{-1}', err < 1e-4)
print(f'  Max FD error: {err:.2e}')

# (d) Gaussian log-likelihood
np.random.seed(42)
mu = np.zeros(3)
samples = np.random.multivariate_normal(mu, A, size=100)
n_s = samples.shape[0]
L = la.cholesky(A)
alpha_vals = sla.cho_solve((L, True), (samples - mu).T)
log_lik = -0.5*(n_s*3*np.log(2*np.pi) + n_s*ld_chol + np.sum(samples @ la.inv(A) * samples))
print(f'\n(d) Gaussian log-likelihood (n=100 samples): {log_lik:.4f}')
print("\nTakeaway: Cholesky log-det is numerically stable and O(n^3/3); direct det() overflows for large n.")

---

## Exercise 7 ★★★ — Gram Matrices and Kernel Regression

**(a)** Given 6 points in $\mathbb{R}^3$, compute the Gram matrix $G = XX^\top$ and verify PSD.

**(b)** Compute kernel matrices for linear, RBF, and polynomial kernels. Verify PSD for each.

**(c)** Apply kernel regression: $\hat{\mathbf{y}}_* = K_{*n}(K_{nn} + 0.1I)^{-1}\mathbf{y}$.

**(d)** Test whether the Laplacian kernel $k(\mathbf{x},\mathbf{z}) = \exp(-\|\mathbf{x}-\mathbf{z}\|)$ is PSD.

**(e)** Explain using Mercer's theorem why the RBF kernel always produces PSD Gram matrices.

In [ ]:
# Exercise 7: Your Solution

np.random.seed(42)
X = np.random.randn(6, 3)  # 6 points in R^3

# (a) Gram matrix
G = None  # YOUR CODE HERE

# (b) Kernels - implement each
def rbf_kernel(X, Y, l=1.0):
    pass  # YOUR CODE HERE

def linear_kernel(X, Y):
    pass  # YOUR CODE HERE

def poly_kernel(X, Y, d=2, c=1.0):
    pass  # YOUR CODE HERE

print(f'G shape: {G.shape if G is not None else None}')
print(f'G PSD: {is_psd(G) if G is not None else None}')

In [ ]:
# Exercise 7: Solution

import scipy.linalg as sla

np.random.seed(42)
X_train = np.random.randn(6, 3)  # 6 training points in R^3
y_train = np.sin(X_train[:, 0]) + 0.1*np.random.randn(6)  # target: sin of first coord
X_test  = np.random.randn(4, 3)  # 4 test points

header('Exercise 7: Gram Matrices and Kernel Regression')

# (a) Gram matrix
G = X_train @ X_train.T
eigs_G = la.eigvalsh(G)
print(f'(a) G = X X^T, shape {G.shape}')
print(f'    Eigenvalues: {eigs_G.round(4)}')
print(f'    rank(G) = {la.matrix_rank(G)} (= rank(X) = {la.matrix_rank(X_train)})')
check_true('(a) G is PSD', np.all(eigs_G >= -1e-10))

# (b) Kernel matrices
def rbf_kernel(X, Y, l=1.0):
    diffs = X[:, None, :] - Y[None, :, :]
    return np.exp(-np.sum(diffs**2, axis=-1) / (2*l**2))

def linear_kernel(X, Y):
    return X @ Y.T

def poly_kernel(X, Y, d=2, c=1.0):
    return (X @ Y.T + c)**d

def laplacian_kernel(X, Y):
    diffs = X[:, None, :] - Y[None, :, :]
    return np.exp(-np.sqrt(np.sum(diffs**2, axis=-1) + 1e-12))

print(f'\n(b) Kernel matrix analysis:')
print(f"{'Kernel':<18} {'min eig':>10} {'PSD':>6}")
print('-'*38)
for kname, K in [('Linear',    linear_kernel(X_train, X_train)),
                  ('RBF (l=1)', rbf_kernel(X_train, X_train)),
                  ('Poly (d=2)',poly_kernel(X_train, X_train)),
                  ('Laplacian', laplacian_kernel(X_train, X_train))]:
    me = la.eigvalsh(K).min()
    print(f'{kname:<18} {me:>10.6f} {str(me >= -1e-10):>6}')

# (c) Kernel regression
reg_param = 0.1
K_nn = rbf_kernel(X_train, X_train)
K_sn = rbf_kernel(X_test, X_train)
K_noisy = K_nn + reg_param * np.eye(6)
alpha = la.solve(K_noisy, y_train)
y_pred = K_sn @ alpha

y_true_test = np.sin(X_test[:, 0])
mse = np.mean((y_pred - y_true_test)**2)
print(f'\n(c) Kernel regression MSE on test: {mse:.4f}')
print(f'    Predictions: {y_pred.round(4)}')
print(f'    True values: {y_true_test.round(4)}')

# (d) Laplacian is also a valid PD kernel (Bochner: exp(-||x||) is PD)
K_lap = laplacian_kernel(X_train, X_train)
min_eig_lap = la.eigvalsh(K_lap).min()
check_true('(d) Laplacian kernel is PSD', min_eig_lap >= -1e-10)

# (e) Explanation
print('\n(e) Mercer theorem: RBF kernel k(x,z) = exp(-||x-z||^2 / 2l^2)')
print('    Fourier transform of this function is a Gaussian (non-negative).')
print('    By Bochner: any stationary kernel with non-negative spectral density is PD.')
print("\nTakeaway: PSD Gram matrices = valid kernels; the reparameterization trick for sampling" \
      " from GP posteriors uses exactly this structure.")

---

## Exercise 8 ★★★ — Cholesky Reparameterization for VAEs

**(a)** Implement `sample_mvn(mu, L, n)` that samples from $\mathcal{N}(\boldsymbol{\mu}, LL^\top)$ using $\mathbf{z} = \boldsymbol{\mu} + L\boldsymbol{\epsilon}$.

**(b)** Verify the sample mean and covariance match the theoretical values.

**(c)** Compute the VAE KL divergence $D_\text{KL}(\mathcal{N}(\boldsymbol{\mu},\Sigma)\|\mathcal{N}(\mathbf{0},I))$ analytically.

**(d)** Verify the KL formula via Monte Carlo estimation.

**(e)** Show that the reparameterization $\mathbf{z} = \boldsymbol{\mu} + L\boldsymbol{\epsilon}$ allows gradient flow through $\boldsymbol{\mu}$ and $L$.

In [ ]:
# Exercise 8: Your Solution

def sample_mvn(mu, L, n_samples):
    # YOUR CODE HERE
    # Sample z ~ N(mu, L L^T) using reparameterization trick
    pass

mu = np.array([1., 2.])
Sigma = np.array([[4., 2.], [2., 3.]])
L = la.cholesky(Sigma)
samples = sample_mvn(mu, L, 100)
print('Sample shape:', samples.shape if samples is not None else None)
print('Sample mean:', samples.mean(axis=0) if samples is not None else None)

In [ ]:
# Exercise 8: Solution

np.random.seed(42)

def sample_mvn(mu, L, n_samples):
    """Sample from N(mu, L L^T) via reparameterization."""
    d = len(mu)
    eps = np.random.randn(d, n_samples)  # isotropic
    return (mu[:, None] + L @ eps).T     # shape (n, d)

header('Exercise 8: VAE Reparameterization Trick')

# (a) + (b)
mu = np.array([1., 2.])
Sigma = np.array([[4., 2.], [2., 3.]])
L = la.cholesky(Sigma)

n_samples = 5000
samples = sample_mvn(mu, L, n_samples)
mu_hat    = samples.mean(axis=0)
Sigma_hat = np.cov(samples.T)

check_close('(b) Sample mean close to mu', mu_hat, mu, tol=0.1)
check_close('(b) Sample cov close to Sigma', Sigma_hat, Sigma, tol=0.3)
print(f'  True mu:    {mu}')
print(f'  Sample mu:  {mu_hat.round(3)}')

# (c) KL divergence analytically
# KL(N(mu, Sigma) || N(0,I)) = 0.5 * (tr(Sigma) + mu^T mu - d - log det Sigma)
d = len(mu)
log_det_Sigma = 2.0 * np.sum(np.log(np.diag(L)))
KL_analytic = 0.5 * (np.trace(Sigma) + mu @ mu - d - log_det_Sigma)
print(f'\n(c) KL (analytic) = {KL_analytic:.6f}')
print(f'    tr(Sigma) = {np.trace(Sigma):.4f}')
print(f'    mu^T mu   = {mu @ mu:.4f}')
print(f'    log det   = {log_det_Sigma:.4f}')

# (d) Monte Carlo KL estimate
# KL(q||p) = E_q[log q(z) - log p(z)]
from scipy.stats import multivariate_normal
log_q = multivariate_normal.logpdf(samples, mean=mu, cov=Sigma)
log_p = multivariate_normal.logpdf(samples, mean=np.zeros(d), cov=np.eye(d))
KL_mc = np.mean(log_q - log_p)
check_close('(d) MC KL close to analytic', KL_mc, KL_analytic, tol=0.05)
print(f'  KL (MC estimate) = {KL_mc:.4f}')

# (e) Gradient flow illustration
# In reparameterized form z = mu + L @ eps:
# dz/d_mu = I (gradient flows to mu)
# dz/d_L  = eps outer product (gradient flows to L)
eps_sample = np.random.randn(d)
z = mu + L @ eps_sample
dz_dmu = np.eye(d)               # Jacobian wrt mu: identity
dz_dL  = np.outer(np.ones(d), eps_sample)  # dz_i/dL_{ij} = eps_j
print(f'\n(e) Reparameterization: z = mu + L @ eps')
print(f'  dz/dmu = I (gradient flows through mu)')
print(f'  dz/dL has shape {dz_dL.shape} (gradient flows through L)')
print(f'  Without reparameterization, z ~ N(mu,Sigma) is a black-box sample -> no gradients')
print("\nTakeaway: Cholesky reparameterization enables backprop through covariance sampling" \
      " by expressing z as a deterministic function of (mu, L, eps).")

---

## What to Review After Finishing

- [ ] Exercise 1: $A^\top A + \lambda I \succ 0$ — why Tikhonov regularization always works
- [ ] Exercise 2: Cholesky algorithm — column-by-column construction; failure = non-PD
- [ ] Exercise 3: Sylvester's criterion — analytical range for parameter-dependent PD
- [ ] Exercise 4: LDLᵀ pivots encode definiteness; avoids square roots
- [ ] Exercise 5: Schur complement = conditional covariance; block matrix inverse
- [ ] Exercise 6: Log-det via Cholesky is stable; gradient is $A^{-1}$
- [ ] Exercise 7: All PSD matrices are Gram matrices; kernel regression uses Cholesky solve
- [ ] Exercise 8: Reparameterization trick: $\mathbf{z} = \boldsymbol{\mu} + L\boldsymbol{\epsilon}$ for differentiable sampling

## References

1. Golub & Van Loan, *Matrix Computations* (4th ed.), Ch. 4
2. Boyd & Vandenberghe, *Convex Optimization*, Ch. 2 (convex sets)
3. Rasmussen & Williams, *Gaussian Processes for Machine Learning*, Ch. 2
4. Kingma & Welling (2013), *Auto-Encoding Variational Bayes* (VAE reparameterization)
5. Martens & Grosse (2015), *Optimizing Neural Networks with Kronecker-factored Approximate Curvature*